<a href="https://colab.research.google.com/github/erizov/Aphorium/blob/main/%D0%92%D1%8B%D0%B7%D0%BE%D0%B2_%D0%B4%D0%B2%D1%83%D1%85_MCP_tools_%D1%87%D0%B5%D1%80%D0%B5%D0%B7_longchain_routing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Для запуска этого скрипта в **Google Colab** нужно учесть два момента:

1. В Colab уже установлен `asyncio`, но событийный цикл (event loop) там уже запущен. Поэтому вместо `asyncio.run()` мы будем использовать `await` напрямую или библиотеку `nest_asyncio`.
2. Нам нужно установить необходимые библиотеки в первой ячейке.

Ниже представлен адаптированный код для ячейки Colab.

### Шаг 1: Установка зависимостей

Создайте новую ячейку и выполните:

In [ ]:
!pip install -U langchain-openai mcp nest_asyncio

---

### Шаг 2: Скрипт для работы в Colab

Скопируйте этот код в следующую ячейку. Вам нужно будет ввести свой API ключ OpenAI в поле ввода, которое появится при запуске.

In [ ]:
import asyncio
import nest_asyncio
from typing import List
from google.colab import widgets
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from mcp import ClientSession
from mcp.client.sse import sse_client
import getpass
import os

# Разрешаем вложенные циклы событий для Colab
nest_asyncio.apply()

# Настройка API ключа
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Введите ваш OpenAI API Key: ")

MCP_SERVERS = {
    "fetch": "https://fetch.mcp.run/sse",
    "time": "https://time.mcp.run/sse"
}

class MCPToolWrapper:
    """Обертка для вызова инструментов MCP сервера через SSE"""
    def __init__(self, server_url):
        self.url = server_url

    async def call_mcp_tool(self, tool_name: str, arguments: dict):
        async with sse_client(self.url) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool(tool_name, arguments)
                # Возвращаем текст из первого блока контента
                return result.content[0].text

# --- Определение инструментов (Tools) ---

@tool
async def get_web_page(url: str):
    """Используй это, если нужно прочитать содержимое веб-страницы или получить актуальные данные из интернета по ссылке."""
    wrapper = MCPToolWrapper(MCP_SERVERS["fetch"])
    return await wrapper.call_mcp_tool("fetch", {"url": url})

@tool
async def get_current_time(timezone: str = "UTC"):
    """Используй это, если нужно узнать точное текущее время или дату. По умолчанию UTC."""
    wrapper = MCPToolWrapper(MCP_SERVERS["time"])
    return await wrapper.call_mcp_tool("get_current_time", {"timezone": timezone})

# --- Основная логика агента ---

llm = ChatOpenAI(model="gpt-4o-mini") # mini быстрее и дешевле для тестов
tools = [get_web_page, get_current_time]
llm_with_tools = llm.bind_tools(tools)

async def process_query(query: str):
    print(f"\n{'='*20}\nЗАПРОС: {query}")

    # 1. Анализ намерения
    ai_msg = await llm_with_tools.ainvoke(query)

    if not ai_msg.tool_calls:
        print("МАРШРУТ: Напрямую в OpenAI (внутренние знания)")
        print(f"ОТВЕТ: {ai_msg.content}")
        return

    # 2. Обработка вызовов инструментов
    print(f"МАРШРУТ: Внешний MCP сервер")
    for tool_call in ai_msg.tool_calls:
        t_name = tool_call["name"]
        t_args = tool_call["args"]
        print(f"Вызов инструмента: {t_name} с аргументами {t_args}")

        # Выбираем функцию по имени
        selected_tool = {"get_web_page": get_web_page, "get_current_time": get_current_time}[t_name]
        tool_output = await selected_tool.ainvoke(t_args)

        # 3. Формирование финального ответа
        final_answer = await llm.ainvoke([
            ("system", "Ты — полезный ассистент. Отвечай кратко на основе предоставленных данных инструментов."),
            ("user", query),
            ai_msg,
            ("tool", str(tool_output), tool_call["id"])
        ])
        print(f"ФИНАЛЬНЫЙ ОТВЕТ: {final_answer.content}")

# Запуск тестов
async def run_tests():
    await process_query("Который час в Нью-Йорке?")
    await process_query("О чем сайт https://www.google.com?")
    await process_query("В каком году Гагарин полетел в космос?")

# В Colab запускаем через await
await run_tests()

---

### Что изменилось для Colab:

* **`nest_asyncio`**: Без этой библиотеки вы бы получили ошибку `RuntimeError: This event loop is already running`, так как Google Colab сам по себе работает внутри асинхронного цикла.
* **`getpass`**: Безопасный ввод API-ключа, чтобы не "светить" его в коде ячейки.
* **`await run_tests()`**: В Colab (и Jupyter) можно вызывать `await` на верхнем уровне, не создавая отдельный `asyncio.run()`.

Хотите, я добавлю в скрипт обработку ошибок, чтобы если один MCP сервер временно недоступен, программа не "падала", а просила OpenAI ответить самостоятельно?